# TinyEarth - Dataset visualisation

Inspects the Phase 2 data pipeline: sample structure, temporal sequences, cloud masking,
channel statistics and the train/val partition.

**Runs on synthetic data by default** - no download required. Switch `DATASET` to
`"earthnet2021"` once you have the real data (see `docs/datasets.md`).

> Results from synthetic data are meaningless. This notebook verifies *plumbing*, not
> forecasting quality.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from tinyearth.config.resolution import ResolvedPaths
from tinyearth.config.schema import DataConfig, LoaderConfig, SyntheticConfig
from tinyearth.datasets import (
    BAND_NAMES,
    RGB_INDICES,
    Split,
    assign_partition,
    build_datamodule,
    summarise_partition,
)
from tinyearth.utils.logging import setup_logging
from tinyearth.utils.paths import cache_dir, data_dir, outputs_dir, project_root
from tinyearth.utils.seed import seed_everything

setup_logging(level="WARNING", rich=False, force=True)
seed_everything(42, deterministic=False)

DATASET = "synthetic"   # or "earthnet2021"
plt.rcParams["figure.dpi"] = 110

## 1. Build the pipeline

The same `DataConfig` a Hydra run composes, so this notebook and a training run agree on
what was configured.

In [ ]:
paths = ResolvedPaths(
    root=project_root(),
    data=data_dir(),
    outputs=outputs_dir(),
    cache=cache_dir(),
    run_dir=outputs_dir() / "notebook",
)

cfg = DataConfig(
    name=DATASET,
    root="synthetic_earthnet2021" if DATASET == "synthetic" else "earthnet2021",
    history_length=4,
    horizon=2,
    cloud_masking=True,
    mask_policy="zero",
    val_fraction=0.25 if DATASET == "synthetic" else 0.1,
    loader=LoaderConfig(batch_size=4, num_workers=0, shuffle=False),
    synthetic=SyntheticConfig(n_cubes=6, n_frames=30, size=64, cloud_fraction=0.25),
)

train = build_datamodule(cfg, paths, split=Split.TRAIN, seed=42)
val = build_datamodule(cfg, paths, split=Split.VAL, seed=42)

print(f"train: {len(train.dataset.cubes):4d} cubes  {len(train.dataset):6d} windows")
print(f"val:   {len(val.dataset.cubes):4d} cubes  {len(val.dataset):6d} windows")
print(f"overlap: {len(set(train.dataset.cubes) & set(val.dataset.cubes))} cubes")

## 2. Sample structure

Verify the contract: `images [T,C,H,W]`, `target [K,C,H,W]`, masks with a singleton
channel axis.

In [ ]:
sample = train.dataset[0]

for key in ("images", "target", "images_mask", "target_mask"):
    t = sample[key]
    print(
        f"{key:14s} {str(tuple(t.shape)):22s} "
        f"min={t.min():.3f} max={t.max():.3f} mean={t.mean():.3f}"
    )

m = sample["metadata"]
print(
    f"\ncube_id={m.cube_id}  start={m.start_index}  "
    f"history={m.history_length}  horizon={m.horizon}  valid={m.valid_fraction:.3f}"
)

## 3. The temporal sequence

History frames followed by target frames, as true colour. The crimson border marks the
forecast boundary.

In [ ]:
def to_rgb(frame, gain=2.5):
    """Convert a [C,H,W] reflectance frame to a displayable RGB array."""
    rgb = frame[list(RGB_INDICES)].permute(1, 2, 0).numpy()
    return np.clip(rgb * gain, 0, 1)


images, target = sample["images"], sample["target"]
n_hist, n_targ = images.shape[0], target.shape[0]

fig, axes = plt.subplots(1, n_hist + n_targ, figsize=(2.0 * (n_hist + n_targ), 2.4))
for i in range(n_hist):
    axes[i].imshow(to_rgb(images[i]))
    axes[i].set_title(f"t-{n_hist - i - 1}", fontsize=9)
for j in range(n_targ):
    ax = axes[n_hist + j]
    ax.imshow(to_rgb(target[j]))
    ax.set_title(f"t+{j + 1}", fontsize=9, color="crimson")
    for spine in ax.spines.values():
        spine.set_edgecolor("crimson")
        spine.set_linewidth(2)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("history  |  forecast target", y=1.04)
plt.tight_layout()
plt.show()

## 4. Spectral channels

EarthNet2021 carries blue, green, red and near-infrared. NIR carries the vegetation
signal, and is why this is not an RGB video task.

In [ ]:
frame = sample["images"][-1]
fig, axes = plt.subplots(1, len(BAND_NAMES) + 1, figsize=(3 * (len(BAND_NAMES) + 1), 3))

for i, name in enumerate(BAND_NAMES):
    im = axes[i].imshow(frame[i], cmap="viridis", vmin=0, vmax=float(frame.max()))
    axes[i].set_title(name)
    axes[i].axis("off")
    plt.colorbar(im, ax=axes[i], fraction=0.046)

axes[-1].imshow(to_rgb(frame))
axes[-1].set_title("true colour")
axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 5. Cloud masking

**`cldmsk == 1` means cloudy**, so validity is `1 - cldmsk`. Getting this backwards trains
the model exclusively on cloud, and does so silently. Bright regions below are usable.

In [ ]:
mask = sample["images_mask"]
fig, axes = plt.subplots(2, n_hist, figsize=(2.0 * n_hist, 4.2))

for i in range(n_hist):
    axes[0, i].imshow(to_rgb(sample["images"][i]))
    axes[0, i].set_title(f"t-{n_hist - i - 1}", fontsize=9)
    axes[1, i].imshow(mask[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"valid {mask[i].mean():.2f}", fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_ylabel("imagery")
axes[1, 0].set_ylabel("valid mask")
plt.tight_layout()
plt.show()

print(f"overall valid fraction: {mask.mean():.3f}")

## 6. Fill policies

What the model sees where data is missing. With a masked loss the choice matters mostly
for the *encoder input* rather than for the target.

In [ ]:
from tinyearth.datasets import MaskPolicy, apply_mask_policy

frame_seq = sample["images"][-1:].clone()
frame_mask = sample["images_mask"][-1:]

fig, axes = plt.subplots(1, len(MaskPolicy), figsize=(3.2 * len(MaskPolicy), 3.2))
for ax, policy in zip(axes, MaskPolicy):
    ax.imshow(to_rgb(apply_mask_policy(frame_seq, frame_mask, policy)[0]))
    ax.set_title(policy.value)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Batch and channel statistics

Per-channel distributions over one batch, excluding masked pixels. Cloud is near-white, so
including masked pixels would bias every mean upward.

In [ ]:
batch = next(iter(train.loader))
print("images", tuple(batch["images"].shape), " target", tuple(batch["target"].shape))

values = torch.cat([batch["images"], batch["target"]], dim=1)
valid = torch.cat([batch["images_mask"], batch["target_mask"]], dim=1).expand_as(values)

fig, ax = plt.subplots(figsize=(7, 3.6))
for c, name in enumerate(BAND_NAMES):
    kept = values[:, :, c][valid[:, :, c] > 0].numpy()
    ax.hist(kept, bins=60, alpha=0.55, label=f"{name} (mean={kept.mean():.3f})")
ax.set_xlabel("reflectance")
ax.set_ylabel("count (valid pixels only)")
ax.legend()
ax.set_title("per-channel distribution")
plt.tight_layout()
plt.show()

## 8. Train/val partition

Hash-based, so it stays stable when cubes are added or removed - the property a seeded
shuffle does not have.

In [ ]:
print(summarise_partition(train.dataset.cubes + val.dataset.cubes, cfg.val_fraction))

ids = [f"cube_{i:05d}" for i in range(5000)]
observed = sum(assign_partition(c, 0.1) is Split.VAL for c in ids) / len(ids)
print(f"requested val_fraction=0.10   observed over 5000 ids = {observed:.4f}")

## 9. Reproducibility

Two independently constructed loaders with the same seed must produce identical batches.

In [ ]:
again = build_datamodule(cfg, paths, split=Split.TRAIN, seed=42)
first, second = next(iter(train.loader)), next(iter(again.loader))

print("images identical:", torch.equal(first["images"], second["images"]))
print("target identical:", torch.equal(first["target"], second["target"]))
print("cube cache:", train.dataset.cache_statistics())

---

## Next

Phase 3 adds the ConvLSTM and temporal-transformer baselines, the training loop, and the
forecast-quality and efficiency metrics. The contract shown above is what they consume.